In [1]:
import pandas as pd
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import plotly.express as px

# Pull in Training CSV

In [2]:
str_project = os.getcwd().split('\\')[4]
print(f'Project: {str_project}')
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

Project: 06_ml_in_python
Task: 01_home_credit_model
Subtask: 01_data_collection


In [3]:
os.getcwd()

'M:\\Risk Management\\Bridger Hyde\\20240521_bridger_internship\\06_ml_in_python\\01_home_credit_model\\01_data_collection'

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

In [23]:
df = pd.read_csv('./input/train.csv')
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,396902,0,Cash loans,F,Y,Y,0,121500,835380,40320.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
1,112096,0,Cash loans,F,N,Y,0,202500,516069,26478.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
2,285821,1,Cash loans,M,Y,Y,1,180000,284400,22469.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
3,376901,0,Cash loans,F,N,Y,0,90000,265536,13685.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,325138,0,Cash loans,F,Y,Y,0,94500,755190,30078.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,217133,0,Revolving loans,M,Y,Y,0,54000,157500,7875.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
124996,169482,0,Cash loans,F,N,N,0,180000,661500,17577.0,...,0,0,0,0,0.0,0.0,0.0,2.0,0.0,2.0
124997,333982,0,Cash loans,F,N,Y,0,90000,239850,24705.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
124998,249620,0,Revolving loans,M,N,Y,0,76500,180000,9000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0


In [24]:
# insert row numbers for easy splitting
rownum = range(len(df))
df.insert(0, 'rownum', rownum)
df.head()

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,0,396902,0,Cash loans,F,Y,Y,0,121500,835380,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
1,1,112096,0,Cash loans,F,N,Y,0,202500,516069,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
2,2,285821,1,Cash loans,M,Y,Y,1,180000,284400,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
3,3,376901,0,Cash loans,F,N,Y,0,90000,265536,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,4,325138,0,Cash loans,F,Y,Y,0,94500,755190,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


# Clean data before splitting

In [25]:
missing = df.isnull().mean() * 100
print(missing[missing > 0])
cols_to_remove = missing[missing > 65].index
dfclean = df.drop(columns=cols_to_remove)
numcols = dfclean.select_dtypes(include=['float64','int64']).columns
catcols = dfclean.select_dtypes(include=['object']).columns
dfclean[numcols] = dfclean[numcols].fillna(dfclean[numcols].median())
for col in catcols:
    dfclean[col] = dfclean[col].fillna(dfclean[col].mode()[0])
dfclean

AMT_ANNUITY                    0.0024
AMT_GOODS_PRICE                0.1056
NAME_TYPE_SUITE                0.4488
OWN_CAR_AGE                   65.9352
OCCUPATION_TYPE               31.4336
                               ...   
AMT_REQ_CREDIT_BUREAU_DAY     13.4040
AMT_REQ_CREDIT_BUREAU_WEEK    13.4040
AMT_REQ_CREDIT_BUREAU_MON     13.4040
AMT_REQ_CREDIT_BUREAU_QRT     13.4040
AMT_REQ_CREDIT_BUREAU_YEAR    13.4040
Length: 66, dtype: float64


,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,0,396902,0,Cash loans,F,Y,Y,0,121500,835380,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
1,1,112096,0,Cash loans,F,N,Y,0,202500,516069,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
2,2,285821,1,Cash loans,M,Y,Y,1,180000,284400,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
3,3,376901,0,Cash loans,F,N,Y,0,90000,265536,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,4,325138,0,Cash loans,F,Y,Y,0,94500,755190,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,124995,217133,0,Revolving loans,M,Y,Y,0,54000,157500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
124996,124996,169482,0,Cash loans,F,N,N,0,180000,661500,...,0,0,0,0,0.0,0.0,0.0,2.0,0.0,2.0
124997,124997,333982,0,Cash loans,F,N,Y,0,90000,239850,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
124998,124998,249620,0,Revolving loans,M,N,Y,0,76500,180000,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0


In [26]:
X = dfclean.drop('TARGET', axis=1)
y = dfclean['TARGET']

In [27]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
model = CatBoostClassifier(iterations = 1000, learning_rate=0.1, depth=6, verbose=0)

In [29]:
categorical_columns = dfclean.select_dtypes(include=['object', 'category']).columns.tolist()
print(categorical_columns)

['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [30]:
model.fit(X_train, y_train, cat_features=categorical_columns)

In [32]:
score = model.score(X_valid, y_valid)
print("model accuracy: ", score)

model accuracy:  0.9186


In [38]:
predictions = model.predict_proba(X_valid)
print('Predicted Proba: ', predictions)

Predicted Proba:  [[0.96766975 0.03233025]
 [0.98070588 0.01929412]
 [0.92441696 0.07558304]
 ...
 [0.74918924 0.25081076]
 [0.95219185 0.04780815]
 [0.97458425 0.02541575]]


In [39]:
proba = model.predict_proba(X)[:, 1]

df['probability_default'] = proba
df

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,probability_default
0,0,396902,0,Cash loans,F,Y,Y,0,121500,835380,...,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.027731
1,1,112096,0,Cash loans,F,N,Y,0,202500,516069,...,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0,0.105439
2,2,285821,1,Cash loans,M,Y,Y,1,180000,284400,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0.607754
3,3,376901,0,Cash loans,F,N,Y,0,90000,265536,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0.054737
4,4,325138,0,Cash loans,F,Y,Y,0,94500,755190,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0.165591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,124995,217133,0,Revolving loans,M,Y,Y,0,54000,157500,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0.088933
124996,124996,169482,0,Cash loans,F,N,N,0,180000,661500,...,0,0,0,0.0,0.0,0.0,2.0,0.0,2.0,0.018572
124997,124997,333982,0,Cash loans,F,N,Y,0,90000,239850,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0.018110
124998,124998,249620,0,Revolving loans,M,N,Y,0,76500,180000,...,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0,0.059462


In [ ]:
dfwrong = df[['']]

# Split Date into 3 different sets

In [32]:
df_train = dfclean.query('0 <= rownum < 87500')
df_train

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,0,396902,0,Cash loans,F,Y,Y,0,121500,835380,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
1,1,112096,0,Cash loans,F,N,Y,0,202500,516069,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
2,2,285821,1,Cash loans,M,Y,Y,1,180000,284400,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
3,3,376901,0,Cash loans,F,N,Y,0,90000,265536,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,4,325138,0,Cash loans,F,Y,Y,0,94500,755190,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87495,87495,347233,1,Cash loans,M,Y,Y,1,202500,436500,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,3.0
87496,87496,415155,0,Cash loans,F,N,Y,1,90000,325908,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
87497,87497,452814,0,Cash loans,M,Y,Y,1,270000,971280,...,0,0,0,0,0.0,0.0,2.0,0.0,0.0,2.0
87498,87498,346541,0,Cash loans,F,N,Y,0,135000,640080,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [33]:
df_valid = dfclean.query('87500 <= rownum < 106250')
df_valid

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
87500,87500,356241,1,Cash loans,M,N,Y,0,220500,816660,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0
87501,87501,443823,0,Cash loans,F,N,N,0,220500,1535553,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
87502,87502,401657,0,Cash loans,M,Y,Y,1,315000,857169,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
87503,87503,145013,0,Cash loans,F,Y,Y,1,112500,697500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
87504,87504,231633,0,Cash loans,M,Y,N,1,225000,805536,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106245,106245,140745,0,Cash loans,F,N,N,1,157500,942300,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
106246,106246,155170,1,Cash loans,F,N,Y,2,81000,418500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
106247,106247,139134,0,Cash loans,F,N,Y,0,112500,1139846,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
106248,106248,188087,0,Cash loans,F,Y,Y,0,166500,472500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [34]:
df_test = dfclean.query('rownum >= 106250')
df_test

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
106250,106250,349548,1,Cash loans,F,Y,Y,1,112500,545040,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,3.0
106251,106251,115330,1,Cash loans,F,N,Y,1,112500,553806,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
106252,106252,156518,0,Cash loans,M,N,Y,0,126000,269982,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,0.0
106253,106253,391420,0,Cash loans,F,Y,Y,0,135000,526491,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
106254,106254,351610,0,Cash loans,F,N,Y,2,126000,331632,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,124995,217133,0,Revolving loans,M,Y,Y,0,54000,157500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
124996,124996,169482,0,Cash loans,F,N,N,0,180000,661500,...,0,0,0,0,0.0,0.0,0.0,2.0,0.0,2.0
124997,124997,333982,0,Cash loans,F,N,Y,0,90000,239850,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
124998,124998,249620,0,Revolving loans,M,N,Y,0,76500,180000,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0


# 1st Model Test

In [47]:
X_train = df_train.drop(columns=['TARGET'])
y_train = df_train['TARGET']

X_valid = df_valid.drop(columns=['TARGET'])
y_valid = df_valid['TARGET']

X_test = df_test.drop(columns=['TARGET'])
y_test = df_test['TARGET']

In [48]:
model = CatBoostClassifier(
    iterations = 1500,
    learning_rate=0.02,
    depth=8,
    verbose=50
)

In [52]:
train_pool = Pool(data=X_train, label=y_train, cat_features=categorical_columns)
valid_pool = Pool(data=X_valid, label=y_valid, cat_features=categorical_columns)

In [53]:
model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

0:	learn: 0.6729234	test: 0.6728810	best: 0.6728810 (0)	total: 123ms	remaining: 3m 3s
50:	learn: 0.2995641	test: 0.2990631	best: 0.2990631 (50)	total: 10.8s	remaining: 5m 5s
100:	learn: 0.2602752	test: 0.2604773	best: 0.2604773 (100)	total: 21.5s	remaining: 4m 57s
150:	learn: 0.2513477	test: 0.2532334	best: 0.2532334 (150)	total: 32.9s	remaining: 4m 53s
200:	learn: 0.2471336	test: 0.2506220	best: 0.2506220 (200)	total: 45.2s	remaining: 4m 52s
250:	learn: 0.2441682	test: 0.2492737	best: 0.2492737 (250)	total: 57.4s	remaining: 4m 45s
300:	learn: 0.2417743	test: 0.2485812	best: 0.2485774 (299)	total: 1m 10s	remaining: 4m 38s
350:	learn: 0.2398584	test: 0.2481638	best: 0.2481610 (349)	total: 1m 22s	remaining: 4m 29s
400:	learn: 0.2380350	test: 0.2479555	best: 0.2479555 (400)	total: 1m 34s	remaining: 4m 18s
450:	learn: 0.2363076	test: 0.2477391	best: 0.2477391 (450)	total: 1m 46s	remaining: 4m 8s
500:	learn: 0.2347189	test: 0.2475530	best: 0.2475529 (499)	total: 1m 58s	remaining: 3m 56s
550

In [54]:
y_valid_pred = model.predict(X_valid)
valid_accuracy = accuracy_score(y_valid, y_valid_pred)
print("Validation Accuracy: ", valid_accuracy)

y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy: ", test_accuracy)

Validation Accuracy:  0.9199466666666667
Test Accuracy:  0.9195733333333334


In [56]:
feature_importances = model.get_feature_importance()
for feature, importance in zip(X_train.columns, feature_importances):
    print(f'{feature}: {importance}')

rownum: 1.5926634560840978
SK_ID_CURR: 1.376731261380298
NAME_CONTRACT_TYPE: 0.4453918485930486
CODE_GENDER: 2.763813063656411
FLAG_OWN_CAR: 1.4432617520979294
FLAG_OWN_REALTY: 0.29622570679341204
CNT_CHILDREN: 0.488349608737065
AMT_INCOME_TOTAL: 1.5118325860587394
AMT_CREDIT: 3.754783427246804
AMT_ANNUITY: 2.716829370226069
AMT_GOODS_PRICE: 3.809131169402788
NAME_TYPE_SUITE: 0.46167773563669334
NAME_INCOME_TYPE: 1.5657652890270934
NAME_EDUCATION_TYPE: 2.5802463060622736
NAME_FAMILY_STATUS: 1.3311325375482372
NAME_HOUSING_TYPE: 0.6023076340179984
REGION_POPULATION_RELATIVE: 1.291560947180951
DAYS_BIRTH: 4.171291966526367
DAYS_EMPLOYED: 2.291192545927628
DAYS_REGISTRATION: 2.0796549612713853
DAYS_ID_PUBLISH: 2.4784844288231915
FLAG_MOBIL: 0.0
FLAG_EMP_PHONE: 0.008864788443103247
FLAG_WORK_PHONE: 0.8807455861674779
FLAG_CONT_MOBILE: 0.001810092629052294
FLAG_PHONE: 0.5862762705321203
FLAG_EMAIL: 0.07033018829510512
OCCUPATION_TYPE: 1.3069356611194132
CNT_FAM_MEMBERS: 0.5430939935005878
R

# 2nd Model Test (removing uneeded columns)

In [57]:
X_train = df_train.drop(columns=['TARGET','rownum','SK_ID_CURR'])
y_train = df_train['TARGET']

X_valid = df_valid.drop(columns=['TARGET','rownum','SK_ID_CURR'])
y_valid = df_valid['TARGET']

X_test = df_test.drop(columns=['TARGET','rownum','SK_ID_CURR'])
y_test = df_test['TARGET']

In [58]:
# adjusting iterations based on previous score
model = CatBoostClassifier(
    iterations = 850,
    learning_rate=0.02,
    depth=8,
    verbose=50
)

In [59]:
train_pool = Pool(data=X_train, label=y_train, cat_features=categorical_columns)
valid_pool = Pool(data=X_valid, label=y_valid, cat_features=categorical_columns)

In [60]:
model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

0:	learn: 0.6722205	test: 0.6721806	best: 0.6721806 (0)	total: 234ms	remaining: 3m 19s
50:	learn: 0.2975143	test: 0.2970210	best: 0.2970210 (50)	total: 11.3s	remaining: 2m 56s
100:	learn: 0.2596416	test: 0.2599759	best: 0.2599759 (100)	total: 21.7s	remaining: 2m 40s
150:	learn: 0.2507620	test: 0.2527962	best: 0.2527962 (150)	total: 34.4s	remaining: 2m 39s
200:	learn: 0.2466342	test: 0.2503498	best: 0.2503498 (200)	total: 47.3s	remaining: 2m 32s
250:	learn: 0.2436371	test: 0.2489926	best: 0.2489926 (250)	total: 59.3s	remaining: 2m 21s
300:	learn: 0.2415000	test: 0.2483251	best: 0.2483147 (299)	total: 1m 10s	remaining: 2m 9s
350:	learn: 0.2396888	test: 0.2478065	best: 0.2478065 (350)	total: 1m 22s	remaining: 1m 57s
400:	learn: 0.2379202	test: 0.2475272	best: 0.2475272 (400)	total: 1m 33s	remaining: 1m 44s
450:	learn: 0.2362104	test: 0.2473537	best: 0.2473537 (450)	total: 1m 45s	remaining: 1m 33s
500:	learn: 0.2344588	test: 0.2471409	best: 0.2471382 (498)	total: 1m 56s	remaining: 1m 21s
5

In [61]:
y_valid_pred = model.predict(X_valid)
valid_accuracy = accuracy_score(y_valid, y_valid_pred)
print("Validation Accuracy: ", valid_accuracy)

y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy: ", test_accuracy)

Validation Accuracy:  0.92016
Test Accuracy:  0.91936


In [62]:
feature_importances = model.get_feature_importance()
for feature, importance in zip(X_train.columns, feature_importances):
    print(f'{feature}: {importance}')

NAME_CONTRACT_TYPE: 0.4680059807630403
CODE_GENDER: 3.0067544224701837
FLAG_OWN_CAR: 1.5003406298751423
FLAG_OWN_REALTY: 0.3023536546649555
CNT_CHILDREN: 0.3408349721745804
AMT_INCOME_TOTAL: 1.6131173783924688
AMT_CREDIT: 3.8145986227351556
AMT_ANNUITY: 2.8780856461931887
AMT_GOODS_PRICE: 4.117806777144479
NAME_TYPE_SUITE: 0.37723869143295974
NAME_INCOME_TYPE: 1.7233387393666362
NAME_EDUCATION_TYPE: 2.5195107086116595
NAME_FAMILY_STATUS: 1.2981405678636202
NAME_HOUSING_TYPE: 0.6775611699854407
REGION_POPULATION_RELATIVE: 1.4224475548076785
DAYS_BIRTH: 4.118241379546655
DAYS_EMPLOYED: 2.9607368414271904
DAYS_REGISTRATION: 1.9138481852666782
DAYS_ID_PUBLISH: 2.4877778581045598
FLAG_MOBIL: 0.0
FLAG_EMP_PHONE: 0.046703208287462744
FLAG_WORK_PHONE: 0.8504918923495861
FLAG_CONT_MOBILE: 0.0020566607914299794
FLAG_PHONE: 0.5135176279720874
FLAG_EMAIL: 0.04178242561475368
OCCUPATION_TYPE: 1.5690946958452228
CNT_FAM_MEMBERS: 0.4188776871738852
REGION_RATING_CLIENT: 0.6690270504174315
REGION_RATI

# 3rd Model Test

In [68]:
#change verbose to see effect
model = CatBoostClassifier(
    iterations = 850,
    learning_rate=0.02,
    depth=8,
    verbose=35
)

In [69]:
model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

0:	learn: 0.6722205	test: 0.6721806	best: 0.6721806 (0)	total: 222ms	remaining: 3m 8s
35:	learn: 0.3365833	test: 0.3359562	best: 0.3359562 (35)	total: 8.34s	remaining: 3m 8s
70:	learn: 0.2735547	test: 0.2733151	best: 0.2733151 (70)	total: 15.7s	remaining: 2m 52s
105:	learn: 0.2579998	test: 0.2585067	best: 0.2585067 (105)	total: 23.9s	remaining: 2m 47s
140:	learn: 0.2519564	test: 0.2535666	best: 0.2535666 (140)	total: 32.2s	remaining: 2m 42s
175:	learn: 0.2483644	test: 0.2513249	best: 0.2513249 (175)	total: 41.2s	remaining: 2m 37s
210:	learn: 0.2459094	test: 0.2499834	best: 0.2499834 (210)	total: 51.6s	remaining: 2m 36s
245:	learn: 0.2438728	test: 0.2491259	best: 0.2491259 (245)	total: 1m	remaining: 2m 28s
280:	learn: 0.2423531	test: 0.2485291	best: 0.2485291 (280)	total: 1m 8s	remaining: 2m 19s
315:	learn: 0.2409125	test: 0.2481240	best: 0.2481240 (315)	total: 1m 17s	remaining: 2m 11s
350:	learn: 0.2396888	test: 0.2478065	best: 0.2478065 (350)	total: 1m 26s	remaining: 2m 2s
385:	learn:

In [70]:
y_valid_pred = model.predict(X_valid)
valid_accuracy = accuracy_score(y_valid, y_valid_pred)
print("Validation Accuracy: ", valid_accuracy)

y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy: ", test_accuracy)

Validation Accuracy:  0.92016
Test Accuracy:  0.91936


In [71]:
feature_importances = model.get_feature_importance()
feature_names = X_train.columns

feature_importance_pairs = list(zip(feature_names, feature_importances))
sorted_feature_importance_pairs = sorted(feature_importance_pairs, key=lambda x: x[1], reverse=True)

for feature, importance in sorted_feature_importance_pairs:
    print(f'{feature}: {importance}')

EXT_SOURCE_3: 13.026539599336253
EXT_SOURCE_2: 11.47538121689546
EXT_SOURCE_1: 6.254244547288362
DAYS_BIRTH: 4.118241379546655
AMT_GOODS_PRICE: 4.117806777144479
AMT_CREDIT: 3.8145986227351556
CODE_GENDER: 3.0067544224701837
DAYS_EMPLOYED: 2.9607368414271904
DAYS_LAST_PHONE_CHANGE: 2.931194572870713
AMT_ANNUITY: 2.8780856461931887
ORGANIZATION_TYPE: 2.624393744137375
NAME_EDUCATION_TYPE: 2.5195107086116595
DAYS_ID_PUBLISH: 2.4877778581045598
DAYS_REGISTRATION: 1.9138481852666782
NAME_INCOME_TYPE: 1.7233387393666362
AMT_INCOME_TOTAL: 1.6131173783924688
OCCUPATION_TYPE: 1.5690946958452228
FLAG_OWN_CAR: 1.5003406298751423
REGION_POPULATION_RELATIVE: 1.4224475548076785
NAME_FAMILY_STATUS: 1.2981405678636202
FLAG_DOCUMENT_3: 1.1669825642358094
HOUR_APPR_PROCESS_START: 1.0857335409952842
DEF_30_CNT_SOCIAL_CIRCLE: 1.048526511232618
FLAG_WORK_PHONE: 0.8504918923495861
AMT_REQ_CREDIT_BUREAU_YEAR: 0.8055232259823176
OBS_60_CNT_SOCIAL_CIRCLE: 0.7686893691754668
REGION_RATING_CLIENT_W_CITY: 0.7358

# Model 4 removing all uneeded columns and changing depth

In [35]:
X_train = df_train.drop(columns=['TARGET','rownum','SK_ID_CURR','FLAG_DOCUMENT_12','FLAG_DOCUMENT_10','FLAG_MOBIL','FLAG_DOCUMENT_4','FLAG_DOCUMENT_2','FLAG_DOCUMENT_7','FLAG_DOCUMENT_17','FLAG_DOCUMENT_15','FLAG_CONT_MOBILE','FLAG_DOCUMENT_20','FLAG_DOCUMENT_19','FLAG_DOCUMENT_11','FLAG_DOCUMENT_21'])
y_train = df_train['TARGET']

X_valid = df_valid.drop(columns=['TARGET','rownum','SK_ID_CURR','FLAG_DOCUMENT_12','FLAG_DOCUMENT_10','FLAG_MOBIL','FLAG_DOCUMENT_4','FLAG_DOCUMENT_2','FLAG_DOCUMENT_7','FLAG_DOCUMENT_17','FLAG_DOCUMENT_15','FLAG_CONT_MOBILE','FLAG_DOCUMENT_20','FLAG_DOCUMENT_19','FLAG_DOCUMENT_11','FLAG_DOCUMENT_21'])
y_valid = df_valid['TARGET']

X_test = df_test.drop(columns=['TARGET','rownum','SK_ID_CURR','FLAG_DOCUMENT_12','FLAG_DOCUMENT_10','FLAG_MOBIL','FLAG_DOCUMENT_4','FLAG_DOCUMENT_2','FLAG_DOCUMENT_7','FLAG_DOCUMENT_17','FLAG_DOCUMENT_15','FLAG_CONT_MOBILE','FLAG_DOCUMENT_20','FLAG_DOCUMENT_19','FLAG_DOCUMENT_11','FLAG_DOCUMENT_21'])
y_test = df_test['TARGET']

In [36]:
train_pool = Pool(data=X_train, label=y_train, cat_features=categorical_columns)
valid_pool = Pool(data=X_valid, label=y_valid, cat_features=categorical_columns)

In [37]:
#changing depth to see effect
model = CatBoostClassifier(
    iterations = 830,
    learning_rate=0.02,
    depth=6,
    verbose=35
)

In [38]:
model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

0:	learn: 0.6730612	test: 0.6730374	best: 0.6730374 (0)	total: 176ms	remaining: 2m 26s
35:	learn: 0.3391094	test: 0.3382413	best: 0.3382413 (35)	total: 5.78s	remaining: 2m 7s
70:	learn: 0.2762115	test: 0.2752073	best: 0.2752073 (70)	total: 10.9s	remaining: 1m 56s
105:	learn: 0.2606863	test: 0.2598246	best: 0.2598246 (105)	total: 16s	remaining: 1m 49s
140:	learn: 0.2549378	test: 0.2544455	best: 0.2544455 (140)	total: 21.2s	remaining: 1m 43s
175:	learn: 0.2521063	test: 0.2519974	best: 0.2519974 (175)	total: 26.5s	remaining: 1m 38s
210:	learn: 0.2503839	test: 0.2507218	best: 0.2507218 (210)	total: 31.9s	remaining: 1m 33s
245:	learn: 0.2490546	test: 0.2498362	best: 0.2498362 (245)	total: 37.7s	remaining: 1m 29s
280:	learn: 0.2479796	test: 0.2492059	best: 0.2492059 (280)	total: 43.1s	remaining: 1m 24s
315:	learn: 0.2470473	test: 0.2487691	best: 0.2487691 (315)	total: 48.4s	remaining: 1m 18s
350:	learn: 0.2462998	test: 0.2484240	best: 0.2484240 (350)	total: 53.7s	remaining: 1m 13s
385:	learn

In [39]:
y_valid_pred = model.predict(X_valid)
valid_accuracy = accuracy_score(y_valid, y_valid_pred)
print("Validation Accuracy: ", valid_accuracy)

y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy: ", test_accuracy)

Validation Accuracy:  0.9202133333333333
Test Accuracy:  0.91952


In [40]:
feature_importances = model.get_feature_importance()
feature_names = X_train.columns

feature_importance_pairs = list(zip(feature_names, feature_importances))
sorted_feature_importance_pairs = sorted(feature_importance_pairs, key=lambda x: x[1], reverse=True)

for feature, importance in sorted_feature_importance_pairs:
    print(f'{feature}: {importance}')

EXT_SOURCE_3: 18.818595456346753
EXT_SOURCE_2: 16.78164097948164
EXT_SOURCE_1: 7.418011057681205
AMT_GOODS_PRICE: 4.322177514058808
AMT_CREDIT: 4.234384110174529
DAYS_BIRTH: 3.903090013176055
NAME_EDUCATION_TYPE: 3.3360612296512144
CODE_GENDER: 2.967514028258597
DAYS_EMPLOYED: 2.4525804103763424
AMT_ANNUITY: 2.2089946291995717
ORGANIZATION_TYPE: 2.087685112754076
DAYS_LAST_PHONE_CHANGE: 1.9414888468363305
DAYS_ID_PUBLISH: 1.8984547856628906
FLAG_OWN_CAR: 1.4841652259996467
NAME_INCOME_TYPE: 1.4764655492984309
DAYS_REGISTRATION: 1.298885341426827
FLAG_DOCUMENT_3: 1.2959907446027188
OCCUPATION_TYPE: 1.1236888748648834
DEF_30_CNT_SOCIAL_CIRCLE: 1.0457034820575404
NAME_FAMILY_STATUS: 1.027638126626132
FLAG_WORK_PHONE: 0.7597087114319682
REGION_RATING_CLIENT_W_CITY: 0.7557667893018376
REGION_POPULATION_RELATIVE: 0.7141828947356571
AMT_INCOME_TOTAL: 0.6382139478192671
DEF_60_CNT_SOCIAL_CIRCLE: 0.5954364573657119
YEARS_BEGINEXPLUATATION_MEDI: 0.5503709774626889
REGION_RATING_CLIENT: 0.5359011

In [41]:
predictions = model.predict_proba(X_valid)
print("Predicted Probability: ", predictions)

Predicted Probability:  [[0.85085184 0.14914816]
 [0.97595952 0.02404048]
 [0.94523549 0.05476451]
 ...
 [0.94999231 0.05000769]
 [0.95552576 0.04447424]
 [0.72930223 0.27069777]]


# Finished with CatBoost Modeling FINAL SCORES: 
# Validation accuracy: 92.02% | Test accuracy: 91.95%

# Logistic Regression Modeling

In [8]:
categorical_columns = dfclean.select_dtypes(include=['object', 'category']).columns.tolist()

In [9]:
dfclean = pd.get_dummies(dfclean, columns=categorical_columns, drop_first=True)

In [10]:
X = dfclean.drop('TARGET', axis = 1)
y = dfclean['TARGET']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_test = scaler.transform(X_test)

In [13]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=100)

In [14]:
model.fit(X_train, y_train)

LogisticRegression(solver='liblinear')

In [15]:
y_pred = model.predict(X_test)

C:\Users\bhyde\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [16]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.40636
Confusion Matrix:
 [[ 8747 14240]
 [  601  1412]]
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.38      0.54     22987
           1       0.09      0.70      0.16      2013

    accuracy                           0.41     25000
   macro avg       0.51      0.54      0.35     25000
weighted avg       0.87      0.41      0.51     25000



In [17]:
# find best model for Logistic Regression
#def eval_model(C, solver, penalty):
#    try:
#        model = LogisticRegression(C=C, solver=solver, penalty=penalty, max_iter=1000)
#        model.fit(X_train, y_train)
#        y_pred = model.predict(X_test)
#        accuracy = accuracy_score(y_test, y_pred)
#        print(f"Accuracy with C={C}, solver={solver}, penalty={penalty}: {accuracy}")
#        print("Classification Report:\n", classification_report(y_test, y_pred))
#        return model, accuracy
#    except Exception as e:
#        print(f"Error with C={C}, solver={solver}, penalty={penalty}: {e}")
#        return None, 0
        

In [18]:
#best_accuracy = 0
#best_model = None
#hyperparams = [
#    (C, solver, penalty)
#    for C in [0.01, 0.1, 1, 10, 100]
#    for solver in ['liblinear', 'newton-cg', 'lbfgs', 'saga']
#    for penalty in ['l2', 'l1']
#    if not (solver in ['newton-cg', 'lbfgs'] and penalty == 'l1')
#]

#for C, solver, penalty in hyperparams:
#    model, accuracy = eval_model(C, solver, penalty)
#    if accuracy > best_accuracy:
#        best_accuracy = accuracy
#        best_model = model

In [17]:
# best model: C= 0.01, solver=liblinear, penalty=l1
model = LogisticRegression(solver='liblinear', C=0.01, penalty='l1', max_iter=1000)

In [18]:
model.fit(X_train, y_train)

LogisticRegression(C=0.01, max_iter=1000, penalty='l1', solver='liblinear')

In [19]:
y_pred = model.predict(X_test)

C:\Users\bhyde\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [20]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.91948
Confusion Matrix:
 [[22987     0]
 [ 2013     0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96     22987
           1       0.00      0.00      0.00      2013

    accuracy                           0.92     25000
   macro avg       0.46      0.50      0.48     25000
weighted avg       0.85      0.92      0.88     25000



C:\Users\bhyde\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bhyde\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bhyde\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [42]:
y_score = model.predict_proba(X)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_score)

fig = px.area(
    x=fpr, y=tpr,
    title=f'ROC CURVE(AUC={auc(fpr, tpr):.4f})',
    width=700, height=500
)
fig.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_xaxes(constrain='domain')
fig.show()

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=4]="Y": Cannot convert 'b'Y'' to float

In [8]:
df_train_log = dfclean.query('0 <= rownum < 87500')
df_train_log

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,0,396902,0,Cash loans,F,Y,Y,0,121500,835380,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
1,1,112096,0,Cash loans,F,N,Y,0,202500,516069,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
2,2,285821,1,Cash loans,M,Y,Y,1,180000,284400,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
3,3,376901,0,Cash loans,F,N,Y,0,90000,265536,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,4,325138,0,Cash loans,F,Y,Y,0,94500,755190,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87495,87495,347233,1,Cash loans,M,Y,Y,1,202500,436500,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,3.0
87496,87496,415155,0,Cash loans,F,N,Y,1,90000,325908,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
87497,87497,452814,0,Cash loans,M,Y,Y,1,270000,971280,...,0,0,0,0,0.0,0.0,2.0,0.0,0.0,2.0
87498,87498,346541,0,Cash loans,F,N,Y,0,135000,640080,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [9]:
df_test_log = dfclean.query('rownum >= 87500')
df_test_log

,rownum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
87500,87500,356241,1,Cash loans,M,N,Y,0,220500,816660,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0
87501,87501,443823,0,Cash loans,F,N,N,0,220500,1535553,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
87502,87502,401657,0,Cash loans,M,Y,Y,1,315000,857169,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
87503,87503,145013,0,Cash loans,F,Y,Y,1,112500,697500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
87504,87504,231633,0,Cash loans,M,Y,N,1,225000,805536,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,124995,217133,0,Revolving loans,M,Y,Y,0,54000,157500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
124996,124996,169482,0,Cash loans,F,N,N,0,180000,661500,...,0,0,0,0,0.0,0.0,0.0,2.0,0.0,2.0
124997,124997,333982,0,Cash loans,F,N,Y,0,90000,239850,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
124998,124998,249620,0,Revolving loans,M,N,Y,0,76500,180000,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0
